In [1]:
pip install torch torchvision h5py numpy matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import argparse
import os
import sys
import time
 
# ── Third-party ───────────────────────────────────────────────────────────────
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms.functional import rotate
from tqdm.auto import tqdm
 
 
# ==============================================================================
# 1.  CLI + Config
# ==============================================================================
 
# ── Environment auto-detection ───────────────────────────────────────────────
def _detect_env() -> str:
    """Return 'kaggle', 'colab', or 'local'."""
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    return "local"
 
 
ENV = _detect_env()
 
# Correct default paths per environment — no manual configuration needed
_DEFAULTS = {
    "kaggle" : {"mnist_root": "/kaggle/working/data/mnist",
                "out":        "/kaggle/working/data/processed"},
    "colab"  : {"mnist_root": "/content/data/mnist",
                "out":        "/content/data/processed"},
    "local"  : {"mnist_root": "./data/mnist",
                "out":        "./data/processed"},
}
 
 
def parse_args() -> argparse.Namespace:
    d = _DEFAULTS[ENV]
    p = argparse.ArgumentParser(
        description=f"Rotated-MNIST Dataset Preparation (HDF5) [{ENV.upper()}]",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--digits",         nargs="+", type=int,  default=[1, 2])
    p.add_argument("--step",           type=int,             default=30,
                   help="Rotation step in degrees. 360 must be divisible by step.")
    p.add_argument("--mnist-root",     type=str,             default=d["mnist_root"])
    p.add_argument("--out",            type=str,             default=d["out"])
    p.add_argument("--compress-level", type=int,             default=4)
    p.add_argument("--chunk-size",     type=int,             default=512)
    p.add_argument("--seed",           type=int,             default=42)
    p.add_argument("--no-verify",      action="store_true")
    p.add_argument("--no-plot",        action="store_true")
 
    # parse_known_args() silently drops the Jupyter kernel flag
    # (-f kernel-xxx.json) that Kaggle / Colab inject into sys.argv.
    # This makes the script safe to %run inside any notebook cell.
    args, unknown = p.parse_known_args()
    if unknown:
        print(f"  [argparse] Ignoring kernel args ({ENV}): {unknown}")
    return args
 
 
def build_config(args: argparse.Namespace) -> dict:
    assert 360 % args.step == 0, f"360 must be divisible by step ({args.step})"
    angles = list(range(0, 360, args.step))
    return {
        "mnist_root"      : args.mnist_root,
        "output_dir"      : args.out,
        "train_file"      : "rotated_mnist_train.h5",
        "test_file"       : "rotated_mnist_test.h5",
        "digits"          : args.digits,
        "rotation_step"   : args.step,
        "angles"          : angles,
        "n_rotations"     : len(angles),
        "img_size"        : 28,
        "chunk_size"      : args.chunk_size,
        "compression"     : "gzip",
        "compression_lvl" : args.compress_level,
        "seed"            : args.seed,
        "verify"          : not args.no_verify,
        "plot"            : not args.no_plot,
    }
 
 
# ==============================================================================
# 2.  MNIST Download & Filter
# ==============================================================================
 
def load_mnist_split(
    root   : str,
    train  : bool,
    digits : list[int],
) -> tuple[np.ndarray, np.ndarray]:
    """
    Download (if needed) and return digit-filtered MNIST as numpy arrays.
 
    Returns
    -------
    images : float32  (N, 28, 28)  pixels in [0, 1]
    labels : int64    (N,)
    """
    dataset = datasets.MNIST(
        root      = root,
        train     = train,
        download  = True,
        transform = transforms.ToTensor(),
    )
 
    mask    = torch.isin(dataset.targets, torch.tensor(digits))
    indices = torch.where(mask)[0]
 
    images = dataset.data[indices].float() / 255.0  # (N, 28, 28)
    labels = dataset.targets[indices].long()
 
    split_name = "train" if train else "test"
    counts = dict(zip(*torch.unique(labels, return_counts=True)))
    counts = {int(k): int(v) for k, v in counts.items()}
    print(f"  [{split_name:5s}] N={len(images):6,d}  class counts={counts}")
 
    return images.numpy(), labels.numpy()
 
 
# ==============================================================================
# 3.  Rotation Engine  (SO(2) orbit sampling)
# ==============================================================================
 
def apply_rotation_orbit(
    images   : np.ndarray,   # float32  (N, H, W)
    labels   : np.ndarray,   # int64    (N,)
    angles   : list[int],
    seed     : int  = 42,
    fill_val : float = 0.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    For each image x generate the discrete SO(2) orbit:
        O(x) = { R_{k * step}[x] }_{k=0}^{K-1}
 
    Resulting arrays are shuffled (random permutation) so that all rotation
    angles are interleaved — important for mini-batch diversity during VAE
    training.
 
    Returns
    -------
    rot_images : float32  (N*K, H, W)
    rot_labels : int64    (N*K,)
    rot_angles : int32    (N*K,)
    """
    N, H, W = images.shape
    K       = len(angles)
 
    rot_images = np.empty((N * K, H, W), dtype=np.float32)
    rot_labels = np.empty(N * K,         dtype=np.int64)
    rot_angles = np.empty(N * K,         dtype=np.int32)
 
    # torchvision.rotate expects (B, C, H, W)
    imgs_t = torch.from_numpy(images).unsqueeze(1)   # (N, 1, H, W)
 
    idx = 0
    for angle in tqdm(angles, desc="  Rotating", unit="angle", leave=True):
        if angle == 0:
            rotated = images                          # identity — no copy needed
        else:
            rotated_t = rotate(
                imgs_t,
                angle         = float(angle),
                interpolation = transforms.InterpolationMode.BILINEAR,
                fill          = [fill_val],
            )                                         # (N, 1, H, W)
            rotated = rotated_t.squeeze(1).numpy()    # (N, H, W)
 
        rot_images[idx : idx + N] = rotated
        rot_labels[idx : idx + N] = labels
        rot_angles[idx : idx + N] = angle
        idx += N
 
    # Shuffle so orbits are not stored in angle-sorted blocks
    rng  = np.random.default_rng(seed=seed)
    perm = rng.permutation(N * K)
 
    return rot_images[perm], rot_labels[perm], rot_angles[perm]
 
 
# ==============================================================================
# 4.  HDF5 Writer
# ==============================================================================
 
def save_to_hdf5(
    filepath   : str,
    images     : np.ndarray,
    labels     : np.ndarray,
    angles     : np.ndarray,
    orig_n     : int,
    split_name : str,
    cfg        : dict,
) -> None:
    """
    Write a rotated-MNIST split to a compressed, self-describing HDF5 file.
 
    HDF5 Schema
    -----------
    /images   float32  (N, 28, 28)   chunked + gzip compressed
    /labels   int64    (N,)
    /angles   int32    (N,)
 
    File attributes store all relevant metadata so the file is self-contained.
    """
    N, H, W = images.shape
    chunk   = min(cfg["chunk_size"], N)
 
    kw = dict(compression=cfg["compression"],
              compression_opts=cfg["compression_lvl"])
 
    with h5py.File(filepath, "w") as f:
 
        # ── Datasets ──────────────────────────────────────────────────────────
        f.create_dataset("images", data=images, dtype=np.float32,
                         chunks=(chunk, H, W), **kw)
        f.create_dataset("labels", data=labels, dtype=np.int64,
                         chunks=(chunk,), **kw)
        f.create_dataset("angles", data=angles, dtype=np.int32,
                         chunks=(chunk,), **kw)
 
        # ── File-level metadata ───────────────────────────────────────────────
        f.attrs["split"]             = split_name
        f.attrs["rotation_angles"]   = cfg["angles"]
        f.attrs["n_rotations"]       = cfg["n_rotations"]
        f.attrs["rotation_step_deg"] = cfg["rotation_step"]
        f.attrs["digit_classes"]     = cfg["digits"]
        f.attrs["original_n"]        = orig_n
        f.attrs["total_n"]           = N
        f.attrs["image_shape"]       = [H, W]
        f.attrs["pixel_range"]       = "[0, 1] float32"
        f.attrs["shuffle"]           = f"random permutation, seed={cfg['seed']}"
        f.attrs["group"]             = f"Z_{cfg['n_rotations']} subset of SO(2)"
        f.attrs["created_by"]        = "symmetry-discovery-project"
 
    size_mb = os.path.getsize(filepath) / 1e6
    print(f"  Saved  → {filepath}")
    print(f"  Shape  : images={images.shape}  labels={labels.shape}  "
          f"angles={angles.shape}")
    print(f"  Size   : {size_mb:.2f} MB  (gzip-{cfg['compression_lvl']})")
 
 
# ==============================================================================
# 5.  HDF5 Verification
# ==============================================================================
 
def verify_hdf5(filepath: str) -> None:
    """Read-back the HDF5 file and print a structured summary + sanity checks."""
    print(f"\n{'='*58}")
    print(f"  VERIFY: {os.path.basename(filepath)}")
    print(f"{'='*58}")
 
    with h5py.File(filepath, "r") as f:
 
        print("\n  [ Datasets ]")
        for name, ds in f.items():
            print(f"    {name:8s} | shape={ds.shape} | dtype={ds.dtype} "
                  f"| chunks={ds.chunks} | compression={ds.compression}")
 
        print("\n  [ Attributes ]")
        for key, val in f.attrs.items():
            print(f"    {key:24s} = {val}")
 
        imgs   = f["images"]
        labels = f["labels"][:]
        angs   = f["angles"][:]
 
        # Pixel range check (sample first 200 rows for speed)
        sample = imgs[:200]
        pmin, pmax = float(sample.min()), float(sample.max())
 
        print("\n  [ Sanity Checks ]")
        print(f"    pixel min / max      : {pmin:.4f} / {pmax:.4f}  "
              f"{'✅' if 0.0 <= pmin and pmax <= 1.0 else '❌ out of range!'}")
        print(f"    unique labels        : {np.unique(labels).tolist()}")
 
        unique_angs, counts = np.unique(angs, return_counts=True)
        print(f"    unique angles        : {unique_angs.tolist()}")
 
        # Check all angles have (roughly) equal representation
        count_std = counts.std()
        print(f"    angle counts std     : {count_std:.1f}  "
              f"{'✅ uniform' if count_std < 5 else '⚠️  uneven'}")
 
 
# ==============================================================================
# 6.  PyTorch Dataset  (for downstream VAE training)
# ==============================================================================
 
class RotatedMNISTDataset(torch.utils.data.Dataset):
    """
    Lazy HDF5-backed PyTorch Dataset for Rotated MNIST.
 
    Each item returns:
        image  : FloatTensor  (1, 28, 28)   channel-first, ready for Conv2d
        label  : LongTensor   ()            digit class
        angle  : LongTensor   ()            rotation angle in degrees
 
    The HDF5 file is opened ONCE per worker process (not per __getitem__),
    avoiding repeated open/close overhead at high throughput.
 
    Usage
    -----
        ds     = RotatedMNISTDataset("rotated_mnist_train.h5")
        loader = DataLoader(ds, batch_size=256, shuffle=True, num_workers=4)
        imgs, labels, angles = next(iter(loader))
    """
 
    def __init__(self, filepath: str, transform=None):
        self.filepath  = filepath
        self.transform = transform
        self._file     = None   # opened lazily in __getitem__
 
        with h5py.File(filepath, "r") as f:
            self._len    = len(f["images"])
            self.angles  = list(f.attrs.get("rotation_angles", []))
            self.digits  = list(f.attrs.get("digit_classes",   []))
 
    # ── Lazy open (one open per worker process) ───────────────────────────────
    def _open(self):
        if self._file is None:
            self._file = h5py.File(self.filepath, "r")
 
    # ── Core API ──────────────────────────────────────────────────────────────
    def __len__(self) -> int:
        return self._len
 
    def __getitem__(self, idx: int):
        self._open()
        image = torch.from_numpy(
            self._file["images"][idx]
        ).unsqueeze(0)                                   # (1, H, W)
        label = torch.tensor(self._file["labels"][idx], dtype=torch.long)
        angle = torch.tensor(self._file["angles"][idx], dtype=torch.long)
 
        if self.transform is not None:
            image = self.transform(image)
 
        return image, label, angle
 
    def __del__(self):
        if self._file is not None:
            try:
                self._file.close()
            except Exception:
                pass
 
    def __repr__(self) -> str:
        return (f"RotatedMNISTDataset("
                f"n={self._len:,d}, digits={self.digits}, "
                f"angles={self.angles})")
 
 
# ==============================================================================
# 7.  Visualizations  (optional)
# ==============================================================================
 
def plot_rotation_orbits(
    h5_path          : str,
    digits           : list[int],
    angles           : list[int],
    n_rows_per_digit : int = 2,
    save_dir         : str = ".",
) -> None:
    """
    Grid plot: rows = samples, columns = rotation angles.
    Illustrates the discrete SO(2) orbit for each digit class.
    """
    with h5py.File(h5_path, "r") as f:
        all_imgs = f["images"][:]
        all_lbls = f["labels"][:]
        all_angs = f["angles"][:]
 
    K          = len(angles)
    total_rows = len(digits) * n_rows_per_digit
 
    fig, axes = plt.subplots(
        total_rows, K,
        figsize=(K * 1.4, total_rows * 1.6),
        facecolor="#0d0d0d",
    )
    fig.suptitle(
        f"SO(2) Orbit: Rotated MNIST  (Δθ = {angles[1]-angles[0]}°, K = {K})",
        color="white", fontsize=13, fontweight="bold", y=1.01,
    )
 
    imgs_t   = torch.from_numpy(all_imgs).unsqueeze(1)  # (N,1,H,W)
    row_idx  = 0
 
    for digit in digits:
        base_mask    = (all_lbls == digit) & (all_angs == 0)
        base_indices = np.where(base_mask)[0]
 
        for sample_num in range(n_rows_per_digit):
            base_img = all_imgs[base_indices[sample_num]]   # (28,28)
 
            for col, angle in enumerate(angles):
                ax = axes[row_idx, col]
 
                img_t = torch.from_numpy(base_img).unsqueeze(0).unsqueeze(0)
                if angle != 0:
                    img_t = rotate(
                        img_t, float(angle),
                        interpolation=transforms.InterpolationMode.BILINEAR,
                        fill=[0.0],
                    )
                rotated_np = img_t.squeeze().numpy()
 
                ax.imshow(rotated_np, cmap="plasma", vmin=0, vmax=1)
                ax.axis("off")
 
                if row_idx == 0:
                    ax.set_title(f"{angle}°", color="white", fontsize=8)
                if col == 0:
                    ax.set_ylabel(
                        f"Digit {digit}", color="white",
                        fontsize=9, rotation=90, labelpad=5,
                    )
 
            row_idx += 1
 
    plt.tight_layout()
    out = os.path.join(save_dir, "orbit_visualization.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
    plt.close()
    print(f"  Orbit grid saved → {out}")
 
 
def plot_pixel_stats_per_angle(
    h5_path  : str,
    angles   : list[int],
    save_dir : str = ".",
) -> None:
    """
    Plot mean pixel intensity and std-dev per rotation angle.
    A flat mean curve confirms near-isometric rotation (rotation conserves energy).
    """
    with h5py.File(h5_path, "r") as f:
        all_imgs = f["images"][:]
        all_angs = f["angles"][:]
 
    means  = [all_imgs[all_angs == a].mean() for a in angles]
    stdevs = [all_imgs[all_angs == a].std()  for a in angles]
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="#0d0d0d")
 
    for ax in axes:
        ax.set_facecolor("#1a1a2e")
        ax.tick_params(colors="white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")
 
    axes[0].plot(angles, means,  "o-", color="#e94560", lw=2, ms=6)
    axes[0].set_title("Mean Pixel Intensity per Angle", color="white", fontsize=11)
    axes[0].set_xlabel("Rotation Angle (°)", color="white")
    axes[0].set_ylabel("Mean", color="white")
    axes[0].set_xticks(angles)
 
    axes[1].plot(angles, stdevs, "s-", color="#e94560", lw=2, ms=6,
                 markerfacecolor="#16213e")
    axes[1].set_title("Pixel Std Dev per Angle", color="white", fontsize=11)
    axes[1].set_xlabel("Rotation Angle (°)", color="white")
    axes[1].set_ylabel("Std Dev", color="white")
    axes[1].set_xticks(angles)
 
    fig.suptitle(
        "Rotation Invariance Check — Pixel Statistics",
        color="white", fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    out = os.path.join(save_dir, "pixel_stats_per_angle.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
    plt.close()
    print(f"  Pixel stats saved → {out}")
    print(f"  Mean range: [{min(means):.5f}, {max(means):.5f}]  "
          f"(flat = near-isometric rotation ✅)")
 
 
# ==============================================================================
# 8.  DataLoader smoke-test
# ==============================================================================
 
def smoke_test_dataloader(h5_path: str, batch_size: int = 256) -> None:
    ds     = RotatedMNISTDataset(h5_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)
    imgs, labels, angles = next(iter(loader))
    print(f"  DataLoader batch → imgs={tuple(imgs.shape)}  "
          f"labels={tuple(labels.shape)}  angles={tuple(angles.shape)}")
    print(f"  {repr(ds)}")
 
 
# ==============================================================================
# 9.  Final Summary
# ==============================================================================
 
def print_summary(cfg: dict) -> None:
    print("\n" + "=" * 62)
    print("  ROTATED MNIST DATASET — FINAL SUMMARY")
    print("=" * 62)
    print(f"  Digits          : {cfg['digits']}")
    print(f"  Rotation group  : Z_{cfg['n_rotations']} ⊂ SO(2),  "
          f"step = {cfg['rotation_step']}°")
    print(f"  Angles          : {cfg['angles']}")
    print(f"  Image shape     : (1, {cfg['img_size']}, {cfg['img_size']})")
    print(f"  Storage format  : HDF5 (gzip-{cfg['compression_lvl']} compressed)")
    print(f"  Output dir      : {cfg['output_dir']}/")
    print()
 
    for fname, split in [(cfg["train_file"], "TRAIN"), (cfg["test_file"], "TEST")]:
        path = os.path.join(cfg["output_dir"], fname)
        if os.path.exists(path):
            size_mb = os.path.getsize(path) / 1e6
            with h5py.File(path, "r") as f:
                N      = f["images"].shape[0]
                orig_n = int(f.attrs.get("original_n", 0))
            print(f"  {split:6s}  | samples={N:8,d} "
                  f"(orig {orig_n:,d} × {cfg['n_rotations']} rotations) "
                  f"| {size_mb:.1f} MB")
 
    print("=" * 62)
    print("\n  ✅  Dataset ready for VAE training  (Task 1 complete)")
    print("  ➡️   Next: 02_vae_training.py\n")
 
 
# ==============================================================================
# 10.  Main Pipeline
# ==============================================================================
 
def main() -> None:
    args = parse_args()
    cfg  = build_config(args)
 
    os.makedirs(cfg["output_dir"], exist_ok=True)
 
    print("\n" + "=" * 62)
    print(f"  Environment     : {ENV.upper()}")
    print(f"  Output dir      : {cfg['output_dir']}")
    print("=" * 62)
    print("  STEP 1: Loading & Filtering MNIST")
    print("=" * 62)
    print(f"  Auto-downloading to: {cfg['mnist_root']}")
    print(f"  Digits: {cfg['digits']}")
 
    train_imgs, train_lbls = load_mnist_split(
        cfg["mnist_root"], train=True,  digits=cfg["digits"]
    )
    test_imgs, test_lbls   = load_mnist_split(
        cfg["mnist_root"], train=False, digits=cfg["digits"]
    )
 
    print("\n" + "=" * 62)
    print("  STEP 2: Building SO(2) Rotation Orbits")
    print("=" * 62)
    print(f"  Angles : {cfg['angles']}")
    print(f"  K      : {cfg['n_rotations']} views per sample")
 
    print("\n  [Training set]")
    t0 = time.time()
    train_rot_imgs, train_rot_lbls, train_rot_angs = apply_rotation_orbit(
        train_imgs, train_lbls, cfg["angles"], seed=cfg["seed"]
    )
    print(f"  Completed in {time.time()-t0:.1f}s  | output shape: {train_rot_imgs.shape}")
 
    print("\n  [Test set]")
    t0 = time.time()
    test_rot_imgs, test_rot_lbls, test_rot_angs = apply_rotation_orbit(
        test_imgs, test_lbls, cfg["angles"], seed=cfg["seed"]
    )
    print(f"  Completed in {time.time()-t0:.1f}s  | output shape: {test_rot_imgs.shape}")
 
    print("\n" + "=" * 62)
    print("  STEP 3: Saving to HDF5")
    print("=" * 62)
 
    save_to_hdf5(
        filepath   = os.path.join(cfg["output_dir"], cfg["train_file"]),
        images     = train_rot_imgs,
        labels     = train_rot_lbls,
        angles     = train_rot_angs,
        orig_n     = len(train_imgs),
        split_name = "train",
        cfg        = cfg,
    )
    save_to_hdf5(
        filepath   = os.path.join(cfg["output_dir"], cfg["test_file"]),
        images     = test_rot_imgs,
        labels     = test_rot_lbls,
        angles     = test_rot_angs,
        orig_n     = len(test_imgs),
        split_name = "test",
        cfg        = cfg,
    )
 
    if cfg["verify"]:
        print("\n" + "=" * 62)
        print("  STEP 4: Verification — Read-Back Check")
        print("=" * 62)
        verify_hdf5(os.path.join(cfg["output_dir"], cfg["train_file"]))
        verify_hdf5(os.path.join(cfg["output_dir"], cfg["test_file"]))
 
        print("\n  DataLoader smoke-test:")
        smoke_test_dataloader(os.path.join(cfg["output_dir"], cfg["train_file"]))
 
    if cfg["plot"]:
        print("\n" + "=" * 62)
        print("  STEP 5: Visualizations")
        print("=" * 62)
        plot_rotation_orbits(
            h5_path          = os.path.join(cfg["output_dir"], cfg["train_file"]),
            digits           = cfg["digits"],
            angles           = cfg["angles"],
            n_rows_per_digit = 2,
            save_dir         = cfg["output_dir"],
        )
        plot_pixel_stats_per_angle(
            h5_path  = os.path.join(cfg["output_dir"], cfg["train_file"]),
            angles   = cfg["angles"],
            save_dir = cfg["output_dir"],
        )
 
    print_summary(cfg)

In [3]:
if __name__ == "__main__":
    main()

  [argparse] Ignoring kernel args (kaggle): ['-f', '/tmp/tmp3ig4p_4q.json', '--HistoryManager.hist_file=:memory:']

  Environment     : KAGGLE
  Output dir      : /kaggle/working/data/processed
  STEP 1: Loading & Filtering MNIST
  Auto-downloading to: /kaggle/working/data/mnist
  Digits: [1, 2]


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 445kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.04MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.04MB/s]


  [train] N=12,700  class counts={1: 6742, 2: 5958}
  [test ] N= 2,167  class counts={1: 1135, 2: 1032}

  STEP 2: Building SO(2) Rotation Orbits
  Angles : [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]
  K      : 12 views per sample

  [Training set]


  Rotating:   0%|          | 0/12 [00:00<?, ?angle/s]

  Completed in 3.3s  | output shape: (152400, 28, 28)

  [Test set]


  Rotating:   0%|          | 0/12 [00:00<?, ?angle/s]

  Completed in 0.4s  | output shape: (26004, 28, 28)

  STEP 3: Saving to HDF5
  Saved  → /kaggle/working/data/processed/rotated_mnist_train.h5
  Shape  : images=(152400, 28, 28)  labels=(152400,)  angles=(152400,)
  Size   : 84.86 MB  (gzip-4)
  Saved  → /kaggle/working/data/processed/rotated_mnist_test.h5
  Shape  : images=(26004, 28, 28)  labels=(26004,)  angles=(26004,)
  Size   : 14.48 MB  (gzip-4)

  STEP 4: Verification — Read-Back Check

  VERIFY: rotated_mnist_train.h5

  [ Datasets ]
    angles   | shape=(152400,) | dtype=int32 | chunks=(512,) | compression=gzip
    images   | shape=(152400, 28, 28) | dtype=float32 | chunks=(512, 28, 28) | compression=gzip
    labels   | shape=(152400,) | dtype=int64 | chunks=(512,) | compression=gzip

  [ Attributes ]
    created_by               = symmetry-discovery-project
    digit_classes            = [1 2]
    group                    = Z_12 subset of SO(2)
    image_shape              = [28 28]
    n_rotations              = 12
    ori